# Phase 6: Pose Estimation
This notebook evaluates the pose estimation accuracy using the ADD metric.

In [1]:
import os
import math
import copy
import open3d as o3d
import numpy as np
from scipy.spatial.transform import Rotation as R
import pandas as pd
import re

def get_rotation_matrix_z(deg):
    """Creates a 3x3 rotation matrix for the Z-axis."""
    rad = math.radians(deg)
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[c, -s, 0], 
                     [s,  c, 0], 
                     [0,  0, 1]])

def merge_multiview_scan(data_dir, initial_pos, initial_angle, box_size, viewpoint_indices=None, T_base2ob_yolo=None, remove_plane=False, do_crop=True):
    """Merges multiple view PCDs and optionally removes points below the detected plane."""
    pcd_files = [f for f in os.listdir(data_dir) if f.lower().endswith('.pcd')]

    def extract_number(filename):
        numbers = re.findall(r'\d+', filename)
        return int(numbers[0]) if numbers else 0
    
    pcd_files.sort(key=extract_number)
    
    if viewpoint_indices is not None:
        pcd_files = [f for f in pcd_files if extract_number(f) in viewpoint_indices]
        
    merged_pcd = o3d.geometry.PointCloud()
    
    if do_crop:
        obb = o3d.geometry.OrientedBoundingBox(
            center=np.array(initial_pos), 
            R=get_rotation_matrix_z(-initial_angle), 
            extent=np.array(box_size)
        )

    print(f"Found {len(pcd_files)} PCD files to merge. Processing...")

    for file_name in pcd_files:
        pcd = o3d.io.read_point_cloud(os.path.join(data_dir, file_name))
   
        if remove_plane:
            # 1. Detect the plane
            plane_model, inliers = pcd.segment_plane(distance_threshold=3.0, ransac_n=3, num_iterations=2000)
            [a, b, c, d] = plane_model

            # 2. Extract all points as a numpy array
            pts = np.asarray(pcd.points)

            # 3. Calculate distance to plane for every point: ax + by + cz + d
            distances = a * pts[:, 0] + b * pts[:, 1] + c * pts[:, 2] + d
            
            # 4. Create an index of points that are ABOVE the plane
            above_plane_indices = np.where(distances > 0.5)[0]
            pcd = pcd.select_by_index(above_plane_indices)

        # 5. Crop to the OBB and merge
        if do_crop:
            merged_pcd += pcd.crop(obb)
        else:
            merged_pcd += pcd

    return merged_pcd

def preprocess_normal(pcd, num_points=False, invert_normals=False, radius=2, max_nn=30):
    """Downsamples, estimates normals, and computes FPFH features."""
    if num_points:
        current_num_points = len(pcd.points)
        if current_num_points >= num_points:
            pcd_down = pcd.farthest_point_down_sample(num_points)
        else:
            pcd_down = o3d.geometry.PointCloud(pcd)
            num_to_pad = num_points - current_num_points
            indices = np.arange(current_num_points)
            pad_indices = np.random.choice(indices, size=num_to_pad, replace=True)
            orig_xyz = np.asarray(pcd.points)
            pad_xyz = orig_xyz[pad_indices]
            jitter = np.random.normal(0, 0.001, pad_xyz.shape)
            pad_xyz += jitter
            final_xyz = np.vstack((orig_xyz, pad_xyz))
            pcd_down.points = o3d.utility.Vector3dVector(final_xyz)
            if pcd.has_normals():
                orig_normals = np.asarray(pcd.normals)
                pad_normals = orig_normals[pad_indices]
                final_normals = np.vstack((orig_normals, pad_normals))
                pcd_down.normals = o3d.utility.Vector3dVector(final_normals)
    else:
        pcd_down = pcd
        
    avg_dist = np.mean(pcd_down.compute_nearest_neighbor_distance())
    pcd_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * radius, max_nn=max_nn))
    normals = np.asarray(pcd_down.normals)
    
    if invert_normals:
        for i in range(len(normals)):
            if normals[i][2] < 0:
                normals[i] *= -1

    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down, o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * 5, max_nn=100))
    
    return pcd_down, fpfh

def run_global_registration_adaptive(source_down, target_down, source_fpfh, target_fpfh):
    max_attempts = 2
    best_fitness = -0.1
    best_inlier_rmse = 100.0
    best_result = None
    best_threshold = None 
    thresholds = [7, 5, 3]

    for attempt in range(max_attempts):
        for thr in thresholds:            
            result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
                source_down, target_down, source_fpfh, target_fpfh, 
                mutual_filter=True,
                max_correspondence_distance=thr,
                estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
                ransac_n=3, 
                checkers=[
                    o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.85),
                    o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(thr)
                ], 
                criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000, 0.99)
            )
            if result.fitness > 0.8 and result.inlier_rmse < best_inlier_rmse:
                best_fitness = result.fitness
                best_inlier_rmse = result.inlier_rmse
                best_result = result
                best_threshold = thr
            if best_fitness > 0.8 and best_inlier_rmse < 3.0: 
                print(f"Excellent Global Fit Found at Threshold {best_threshold}")
                return best_result, best_threshold
            
    if best_result is None:
        print("Warning: RANSAC could not find a fit above 0.85 fitness.")
        return result, thr

    print(f"RANSAC Finished. Best Threshold: {best_threshold} | Fitness: {best_fitness:.4f}")
    return best_result, best_threshold

def run_local_refinement_adaptive(source, target, initial_trans=None, best_ransac_thr=10, method="point_to_plane"):
    if initial_trans is None:
        initial_trans = np.eye(4)
    multipliers = [1.0, 0.5, 0.2]
    thresholds = [best_ransac_thr * m for m in multipliers]
    best_result = None
    best_inlier_rmse = float('inf')
    
    if method == "point_to_plane":
        estimation_method = o3d.pipelines.registration.TransformationEstimationPointToPlane()
        print("Using Point-to-Plane ICP")
    else:
        estimation_method = o3d.pipelines.registration.TransformationEstimationPointToPoint()
        print("Using Point-to-Point ICP")
    
    print(f"{'Threshold':<12} | {'Fitness':<12} | {'RMSE':<12}")
    print("-" * 45)
    for thr in thresholds:
        reg_icp = o3d.pipelines.registration.registration_icp(
            source, target, thr, initial_trans,
            estimation_method,
            o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=200)
        )
        print(f"{thr:<12.2f} | {reg_icp.fitness:<12.4f} | {reg_icp.inlier_rmse:<12.4f}")
        if reg_icp.fitness > 0.85 and reg_icp.inlier_rmse < best_inlier_rmse:
            best_inlier_rmse = reg_icp.inlier_rmse
            best_result = reg_icp
    return best_result if best_result is not None else reg_icp

def calculate_add(source_cloud, T_est, T_gt):
    """
    Calculates the Average Distance of Model Points (ADD) using the L2 norm.
    ADD = 1/|M| sum ||(T_est * x) - (T_gt * x)||_2
    """
    points = np.asarray(source_cloud.points)
    ones = np.ones((points.shape[0], 1))
    points_homo = np.hstack([points, ones])
    
    # Apply Estimated Transformation
    est_points = (T_est @ points_homo.T).T[:, :3]
    
    # Apply Ground Truth Transformation
    gt_points = (T_gt @ points_homo.T).T[:, :3]
    
    # Calculate L2 norm (Euclidean distance) for each point
    distances = np.linalg.norm(est_points - gt_points, axis=1)
    
    # ADD is the mean of these distances
    add_metric = np.mean(distances)
    
    return add_metric


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


### 1. Configuration

In [31]:
# --- 1. Configuration ---
NUMBER_OF_POINTS = 40000
EXPERIMENT = "test_8_simulation2"
WORKPIECE = "TH0011AV"

SOURCE_PATH = f"workpiece/{WORKPIECE}/workpiece.stl" # CAD STL model
DATA_DIR = f"pcd_data/testing_data/{EXPERIMENT}/{WORKPIECE}" # Multiview scans

# Manually set which viewpoints to merge. Leave as None to process all.
VIEWPOINT_INDICES = [] 

# Dummy Initialization for Simulation Data
T_base2ob_yolo = np.eye(4)
YOLO_POS = [5.0, -3.0, 2.0]  # Off by 5mm in X, 3mm in Y
YOLO_ANGLE = 5.0               # Off by 5 degrees
CROP_BOX = [1000, 1000, 1000] # Large crop box to avoid cutting off dummy data

# Ground Truth Pose (Currently set to dummy identity matrix 0,0,0,0,0,0)
T_gt = np.eye(4)


### 2. Data Preparation & Merging

In [ ]:
# --- 2. Data Preparation ---
mesh = o3d.io.read_triangle_mesh(SOURCE_PATH)
mesh.compute_vertex_normals()

# CAD Model point cloud
source_cloud = mesh.sample_points_uniformly(number_of_points=NUMBER_OF_POINTS)

# Load and Merge specific viewpoints
full_target_cloud = merge_multiview_scan(DATA_DIR, YOLO_POS, YOLO_ANGLE, CROP_BOX, viewpoint_indices=VIEWPOINT_INDICES, T_base2ob_yolo=T_base2ob_yolo, remove_plane=False, do_crop=False)

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0, origin=[0, 0, 0])
# o3d.visualization.draw_geometries([full_target_cloud, mesh, world_frame], window_name="Merged Target Cloud")

# Manual Initial Guess
T_extra = np.eye(4)
r = R.from_euler('z', 0, degrees=True)
T_extra[:3, :3] = r.as_matrix()
T_initial_guess = T_base2ob_yolo @ T_extra

# Preprocess Normal and FPFH
source_cloud_transformed = copy.deepcopy(source_cloud).transform(T_initial_guess)
source_down, source_fpfh = preprocess_normal(source_cloud_transformed)
target_down, target_fpfh = preprocess_normal(full_target_cloud, num_points=40000, invert_normals=True)

source_down.paint_uniform_color([1, 0, 0])
target_down.paint_uniform_color([0, 0.651, 0.929])
# o3d.visualization.draw_geometries([target_down, source_down], window_name="Processed Initial Target vs Source")


Found 432 PCD files to merge. Processing...


### 2.5 Initial Guess ADD Metric (Before Registration)

In [53]:
# --- 2.5 Initial ADD ACCURACY MEASUREMENT ---
print("="*30)
initial_add_value = calculate_add(source_cloud, T_initial_guess, T_gt)
print(f"Initial Pose Accuracy (ADD) before registration: {initial_add_value:.4f} mm")
print("="*30)


Initial Pose Accuracy (ADD) before registration: 0.0000 mm


### 3. Global Alignment (RANSAC)

In [ ]:
# --- 3. Global Alignment (RANSAC) ---
print("Step 2: Running RANSAC Global Registration...")
ransac_res, best_thr = run_global_registration_adaptive(source_down, target_down, source_fpfh, target_fpfh)
print(ransac_res)

# Visualize RANSAC result
source_temp = copy.deepcopy(source_down)
source_temp.transform(ransac_res.transformation)
source_temp.paint_uniform_color([1, 0, 0])
target_down.paint_uniform_color([0, 0.651, 0.929])
# o3d.visualization.draw_geometries([source_temp, target_down], window_name="RANSAC Result")


Step 2: Running RANSAC Global Registration...
[Open3D WARNING] Too few correspondences (19) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (19) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (19) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (19) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (19) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (19) after mutual filter, fall back to original correspondences.
RegistrationResult with fitness=0.000000e+00, inlier_rmse=0.000000e+00, and correspondence_set size of 0
Access transformation to get result.


### 4. Local Alignment (ICP) & ADD Metric

In [55]:
# --- 4. Local Alignment (ICP) ---
print("Step 3: Running ICP Local Refinement...")
icp_res = run_local_refinement_adaptive(source_down, target_down, method="point_to_plane") #icp only
print(icp_res)

# --- 5. Extract Final Results ---
fine_correction_transformation = icp_res.transformation 
T_est = np.dot(fine_correction_transformation, T_initial_guess) 

print("="*30)
print(f"Estimated Position: {T_est[:3, 3]}")
print(f"Estimated Orientation Matrix:\n{T_est[:3, :3]}")
print(f"Fitness: {icp_res.fitness:.4f}")
print(f"RMSE: {icp_res.inlier_rmse:.4f}")

# --- 6. ADD ACCURACY MEASUREMENT ---
print("="*30)
add_value = calculate_add(source_cloud, T_est, T_gt)
print(f"Pose Estimation Accuracy (ADD): {add_value:.4f} mm")

# --- 7. Final Visualization ---
source_est = copy.deepcopy(source_cloud).transform(T_est)
source_est.paint_uniform_color([1, 0, 0])         # Red: Estimated Pose

source_gt = copy.deepcopy(source_cloud).transform(T_gt)
source_gt.paint_uniform_color([0, 1, 0])         # Green: Ground Truth Pose

full_target_cloud.paint_uniform_color([0, 0.65, 0.93])   # Blue: Scanned Data

o3d.visualization.draw_geometries([source_est, source_gt, full_target_cloud], window_name=f"Pose Estimation (ADD: {add_value:.4f})")


Step 3: Running ICP Local Refinement...
Using Point-to-Plane ICP
Threshold    | Fitness      | RMSE        
---------------------------------------------
10.00        | 0.9375       | 1.8679      
5.00         | 0.8970       | 1.0537      
2.00         | 0.8639       | 0.8157      
RegistrationResult with fitness=8.639250e-01, inlier_rmse=8.157409e-01, and correspondence_set size of 34557
Access transformation to get result.
Estimated Position: [ 1.36154587  2.03238529 -0.90639037]
Estimated Orientation Matrix:
[[ 0.99987593 -0.01153041 -0.01073162]
 [ 0.01147769  0.99992182 -0.00496162]
 [ 0.01078799  0.00483783  0.9999301 ]]
Fitness: 0.8639
RMSE: 0.8157
Pose Estimation Accuracy (ADD): 2.4531 mm


### 5. Batch Processing (Multiple Workpieces)
Run the pipeline on an entire list of workpieces automatically without popping up visualization windows.

In [104]:
# ==========================================
# BATCH PROCESSING MULTIPLE WORKPIECES
# ==========================================
import time
import numpy as np
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R

# Define the list of workpieces you want to evaluate
# BATCH_WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV", "TH0041AV", "TH0042AV", "TH0051AV", "TH0052AV", "TH0061AV", "TH0062AV", "TH0071AV", "TH0072AV"]
ENABLE_VISUALIZATION = True # Set to False if you want it to run headlessly
USE_RANSAC = False # Set to True to use RANSAC + ICP, False for ICP only
EXPERIMENT = "test_8_simulation2"
NUMBER_OF_POINTS = 40000
FPS_NUMBER = 7500

# Dummy Initialization for Simulation Data
YOLO_POS = [5.0, -3.0, 2.0]
YOLO_ANGLE = 5
CROP_BOX = [1000, 1000, 1000]

T_base2ob_yolo = np.eye(4)
T_base2ob_yolo[:3, :3] = R.from_euler('z', YOLO_ANGLE, degrees=True).as_matrix()
T_base2ob_yolo[:3, 3] = YOLO_POS
T_gt = np.eye(4)


BATCH_WORKPIECES = ["TH0011AV"] 
# VIEWPOINT_INDICES = [67]
# GRID LIKE
# VIEWPOINT_INDICES = [80]
# VIEWPOINT_INDICES = [80, 84, 88, 92] 
VIEWPOINT_INDICES = [80, 82, 84, 86, 88, 90, 92, 94]
# VIEWPOINT_INDICES = [80, 81, 85, 86, 87, 89, 80, 91, 93, 94, 95, 97]

batch_results = []

print(f"Starting batch pose estimation for {len(BATCH_WORKPIECES)} workpieces...")

for wp in BATCH_WORKPIECES:
    print(f"\n{'='*40}")
    print(f"Processing Workpiece: {wp}")
    print(f"{'='*40}")
    
    start_time = time.time()
    # Lock the random seed so that any point cloud padding jitter is 100% identical every run
    np.random.seed(42)
    o3d.utility.random.seed(42)
    
    # Paths
    source_path = f"workpiece/{wp}/workpiece.stl"
    data_dir = f"pcd_data/testing_data/{EXPERIMENT}/{wp}"
    
    if not os.path.exists(source_path) or not os.path.exists(data_dir):
        print(f"Skipping {wp} - Missing STL or PCD folder!")
        continue
        
    # 1. Load CAD
    mesh = o3d.io.read_triangle_mesh(source_path)
    mesh.compute_vertex_normals()
    source_cloud = mesh.sample_points_uniformly(number_of_points=NUMBER_OF_POINTS)
    
    # 2. Load Scans
    full_target_cloud = merge_multiview_scan(data_dir, YOLO_POS, YOLO_ANGLE, CROP_BOX, viewpoint_indices=VIEWPOINT_INDICES, T_base2ob_yolo=T_base2ob_yolo, remove_plane=False, do_crop=False)
    # FPS downsample after merging (only if above target count)
    # full_target_cloud = full_target_cloud.voxel_down_sample(voxel_size=1)  # mm
    if len(full_target_cloud.points) == 0:
         print(f"Skipping {wp} - No points found in target cloud!")
         continue
    if len(full_target_cloud.points) > FPS_NUMBER:
        full_target_cloud = full_target_cloud.farthest_point_down_sample(FPS_NUMBER)
    print("Merged point cloud: ", full_target_cloud)
         
    # 3. Initial Guess
    T_extra = np.eye(4)
    r = R.from_euler('z', 0, degrees=True)
    T_extra[:3, :3] = r.as_matrix()
    T_initial_guess = T_base2ob_yolo @ T_extra
    
    # 4. Preprocess
    source_cloud_transformed = copy.deepcopy(source_cloud).transform(T_initial_guess)
    source_down, source_fpfh = preprocess_normal(source_cloud_transformed)
    target_down, target_fpfh = preprocess_normal(full_target_cloud, invert_normals=True)
    o3d.visualization.draw_geometries([full_target_cloud])
    # --- VISUALIZE INITIAL OFFSET (BEFORE ICP) ---
    if ENABLE_VISUALIZATION:
        source_initial = copy.deepcopy(source_cloud_transformed)
        source_initial.paint_uniform_color([1, 0.5, 0])  # Orange: YOLO Initial Guess
        target_vis = copy.deepcopy(full_target_cloud)
        target_vis.paint_uniform_color([0, 0.65, 0.93])  # Blue: Scanned Data
        # o3d.visualization.draw_geometries([source_initial, target_vis], window_name=f"{wp} - 1. INITIAL YOLO GUESS (Orange) vs SCAN (Blue)")
    
    
    # 5. Global (RANSAC) - Optional
    if USE_RANSAC:
        ransac_res, best_thr = run_global_registration_adaptive(source_down, target_down, source_fpfh, target_fpfh)
        icp_initial = ransac_res.transformation
        
        if ENABLE_VISUALIZATION:
            source_ransac = copy.deepcopy(source_cloud_transformed).transform(icp_initial)
            source_ransac.paint_uniform_color([1, 0.5, 0])  # Orange: RANSAC Result
            target_vis = copy.deepcopy(full_target_cloud)
            target_vis.paint_uniform_color([0, 0.65, 0.93])  # Blue: Scanned Data
            o3d.visualization.draw_geometries([source_ransac, target_vis], window_name=f"{wp} - 1.5 RANSAC RESULT (Orange) vs SCAN (Blue)")
    else:
        icp_initial = np.eye(4)
        
    # 6. Local (ICP) - USING FIXED THRESHOLD

    FIXED_ICP_THRESHOLD = 2.0
    icp_res = o3d.pipelines.registration.registration_icp(
        source_down, target_down, FIXED_ICP_THRESHOLD, icp_initial,
        o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=200)
    )
    
    # 7. Evaluate
    T_est = np.dot(icp_res.transformation, T_initial_guess)
    add_value = calculate_add(source_cloud, T_est, T_gt)
    
    process_time = time.time() - start_time
    
    if ENABLE_VISUALIZATION:
        source_est = copy.deepcopy(source_cloud).transform(T_est)
        source_est.paint_uniform_color([1, 0, 0])         # Red: Estimated Pose
        source_gt = copy.deepcopy(source_cloud).transform(T_gt)
        source_gt.paint_uniform_color([0, 1, 0])         # Green: Ground Truth Pose
        full_target_cloud.paint_uniform_color([0, 0.65, 0.93])   # Blue: Scanned Data
        o3d.visualization.draw_geometries([source_est, source_gt, full_target_cloud], window_name=f"{wp} - 2. FINAL ICP RESULT (ADD: {add_value:.4f})")
    
    print(f"--> [RESULT] {wp} | ADD: {add_value:.4f} mm | Time: {process_time:.1f}s")
    
    batch_results.append({
        'Workpiece': wp,
        'ADD_Accuracy_mm': add_value,
        'Fitness': icp_res.fitness,
        'RMSE': icp_res.inlier_rmse,
        'Processing_Time_s': process_time
    })

# Summary
if batch_results:
    df_results = pd.DataFrame(batch_results)
    print("\n\n" + "="*50)
    print("BATCH PROCESSING SUMMARY")
    print("="*50)
    display(df_results)
    
    mean_add = df_results['ADD_Accuracy_mm'].mean()
    print(f"\nAverage ADD across {len(batch_results)} workpieces: {mean_add:.4f} mm")
    
    # Optional: Save to CSV
    # df_results.to_csv("batch_pose_estimation_results.csv", index=False)
    # print("Saved results to batch_pose_estimation_results.csv")


Starting batch pose estimation for 1 workpieces...

Processing Workpiece: TH0011AV
Found 8 PCD files to merge. Processing...
Merged point cloud:  PointCloud with 7500 points.
--> [RESULT] TH0011AV | ADD: 0.5616 mm | Time: 1.7s


BATCH PROCESSING SUMMARY


,Workpiece,ADD_Accuracy_mm,Fitness,RMSE,Processing_Time_s
0,TH0011AV,0.561649,0.786325,1.199624,1.724059



Average ADD across 1 workpieces: 0.5616 mm
